# Sankey diagrams from `sankey_table_relationships.csv`

This notebook renders Sankey diagrams from a CSV whose columns follow the pattern `level<N>_table` plus a numeric `count` column.

Design goals:

* **Level-agnostic.** The list of levels is discovered at runtime from the column names (regex `^level(\d+)_table$`). Drop a column from the CSV and the notebook keeps working.
* **Per-level diagrams.** One Sankey per *adjacent* level pair (`level1 → level2`, `level2 → level3`, …) is rendered.
* **Combined diagram.** A full multi-stage Sankey across all detected levels is also rendered.
* **Two renderers.** A static `matplotlib` image (saved to disk as PNG) and an interactive `plotly` figure (saved as HTML).

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.path import Path as MplPath
import plotly.graph_objects as go

CSV_PATH = Path('sankey_table_relationships.csv')
OUT_DIR = Path('sankey_output')
OUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(CSV_PATH)
df.head()

## Detect level columns dynamically

Any column matching `level<N>_table` is treated as a stage in the flow, sorted by `N`. Anything else (e.g. `count`) is ignored. If a level is missing from the CSV, the remaining levels still form a valid chain of adjacent pairs.

In [ ]:
LEVEL_RE = re.compile(r'^level(\d+)_table$')
COUNT_COL = 'count'

def detect_levels(frame: pd.DataFrame) -> list[str]:
    matches = []
    for col in frame.columns:
        m = LEVEL_RE.match(col)
        if m:
            matches.append((int(m.group(1)), col))
    matches.sort(key=lambda t: t[0])
    return [col for _, col in matches]

level_cols = detect_levels(df)
assert COUNT_COL in df.columns, f"Expected a '{COUNT_COL}' column"
assert len(level_cols) >= 2, 'Need at least two level columns to draw a Sankey'
level_cols

In [ ]:
def adjacent_flows(frame: pd.DataFrame, src: str, dst: str, count_col: str = COUNT_COL) -> pd.DataFrame:
    """Aggregate counts for each (src, dst) pair, dropping rows where either side is missing."""
    sub = frame[[src, dst, count_col]].dropna(subset=[src, dst])
    return (sub.groupby([src, dst], as_index=False)[count_col]
               .sum()
               .rename(columns={src: 'source', dst: 'target', count_col: 'value'}))

adjacent_flows(df, level_cols[0], level_cols[1]).head()

## Static renderer (matplotlib)

Draws each stage as a column of node rectangles, with cubic-bezier ribbons connecting them. The ribbon width is proportional to the flow value.

In [ ]:
def _stage_layout(flows: pd.DataFrame, gap_ratio: float = 0.02):
    """Compute y positions for source and target nodes for a single stage."""
    src_totals = flows.groupby('source')['value'].sum().sort_values(ascending=False)
    dst_totals = flows.groupby('target')['value'].sum().sort_values(ascending=False)

    def positions(totals: pd.Series):
        total = totals.sum()
        gap = total * gap_ratio
        pos = {}
        cursor = 0.0
        for name, val in totals.items():
            pos[name] = (cursor, cursor + val)  # (y0, y1)
            cursor += val + gap
        height = cursor - gap if totals.size else 0
        return pos, height

    src_pos, src_h = positions(src_totals)
    dst_pos, dst_h = positions(dst_totals)
    return src_pos, dst_pos, max(src_h, dst_h)

def _bezier_ribbon(ax, x0, x1, y0a, y0b, y1a, y1b, color, alpha=0.4):
    """Draw a filled cubic-bezier ribbon between two vertical line segments."""
    cx0, cx1 = x0 + (x1 - x0) * 0.5, x0 + (x1 - x0) * 0.5
    verts = [
        (x0, y0a),
        (cx0, y0a), (cx1, y1a), (x1, y1a),  # top edge
        (x1, y1b),
        (cx1, y1b), (cx0, y0b), (x0, y0b),  # bottom edge
        (x0, y0a),
    ]
    codes = [MplPath.MOVETO,
             MplPath.CURVE4, MplPath.CURVE4, MplPath.CURVE4,
             MplPath.LINETO,
             MplPath.CURVE4, MplPath.CURVE4, MplPath.CURVE4,
             MplPath.CLOSEPOLY]
    patch = mpatches.PathPatch(MplPath(verts, codes), facecolor=color, edgecolor='none', alpha=alpha)
    ax.add_patch(patch)

def _color_map(names, cmap_name='tab20'):
    cmap = plt.get_cmap(cmap_name)
    return {n: cmap(i % cmap.N) for i, n in enumerate(sorted(set(names)))}

def draw_static_sankey(flows: pd.DataFrame, title: str, ax=None, node_width=0.04, x0=0.0, x1=1.0):
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(10, 6))
    src_pos, dst_pos, height = _stage_layout(flows)
    colors = _color_map(list(src_pos) + list(dst_pos))

    # node rectangles
    for name, (y0, y1) in src_pos.items():
        ax.add_patch(mpatches.Rectangle((x0, y0), node_width, y1 - y0, color=colors[name]))
        ax.text(x0 - 0.005, (y0 + y1) / 2, name, ha='right', va='center', fontsize=9)
    for name, (y0, y1) in dst_pos.items():
        ax.add_patch(mpatches.Rectangle((x1 - node_width, y0), node_width, y1 - y0, color=colors[name]))
        ax.text(x1 + 0.005, (y0 + y1) / 2, name, ha='left', va='center', fontsize=9)

    # ribbons — consume each node's vertical extent proportionally
    src_cursor = {n: y0 for n, (y0, _) in src_pos.items()}
    dst_cursor = {n: y0 for n, (y0, _) in dst_pos.items()}
    for _, row in flows.iterrows():
        v = row['value']
        s, t = row['source'], row['target']
        y0a = src_cursor[s]; y0b = y0a + v; src_cursor[s] = y0b
        y1a = dst_cursor[t]; y1b = y1a + v; dst_cursor[t] = y1b
        _bezier_ribbon(ax, x0 + node_width, x1 - node_width, y0a, y0b, y1a, y1b, colors[s])

    ax.set_xlim(x0 - 0.18, x1 + 0.18)
    ax.set_ylim(-height * 0.05, height * 1.05)
    ax.invert_yaxis()
    ax.set_axis_off()
    ax.set_title(title)
    if standalone:
        plt.tight_layout()
        return fig

## Interactive renderer (plotly)

For a single adjacent pair the function builds a 2-stage Sankey; the same function handles the combined multi-stage view by passing the full ordered list of level columns.

In [ ]:
def build_interactive_sankey(frame: pd.DataFrame, levels: list[str], title: str) -> go.Figure:
    """Build a Sankey across an ordered list of level columns (2+ entries)."""
    assert len(levels) >= 2
    # Stage-scoped node labels avoid collisions when the same table name appears at different levels.
    node_labels: list[str] = []
    node_index: dict[tuple[int, str], int] = {}
    for stage_idx, col in enumerate(levels):
        for name in frame[col].dropna().unique():
            key = (stage_idx, name)
            if key not in node_index:
                node_index[key] = len(node_labels)
                node_labels.append(name)

    src, tgt, val = [], [], []
    for i in range(len(levels) - 1):
        flows = adjacent_flows(frame, levels[i], levels[i + 1])
        for _, row in flows.iterrows():
            src.append(node_index[(i, row['source'])])
            tgt.append(node_index[(i + 1, row['target'])])
            val.append(row['value'])

    fig = go.Figure(go.Sankey(
        arrangement='snap',
        node=dict(label=node_labels, pad=18, thickness=18, line=dict(color='black', width=0.4)),
        link=dict(source=src, target=tgt, value=val),
    ))
    fig.update_layout(title_text=title, font_size=11, height=520)
    return fig

## Per-level Sankey diagrams

One static figure and one interactive figure for each adjacent pair of detected levels.

In [ ]:
pair_results = []
for i in range(len(level_cols) - 1):
    src_col, dst_col = level_cols[i], level_cols[i + 1]
    flows = adjacent_flows(df, src_col, dst_col)
    title = f'{src_col} \u2192 {dst_col}'

    fig = draw_static_sankey(flows, title=title)
    png_path = OUT_DIR / f'sankey_static_{src_col}_to_{dst_col}.png'
    fig.savefig(png_path, dpi=150, bbox_inches='tight')
    plt.show()

    ifig = build_interactive_sankey(df, [src_col, dst_col], title)
    html_path = OUT_DIR / f'sankey_interactive_{src_col}_to_{dst_col}.html'
    ifig.write_html(html_path)
    ifig.show()

    pair_results.append({'pair': title, 'png': str(png_path), 'html': str(html_path), 'rows': len(flows)})

pd.DataFrame(pair_results)

## Combined Sankey across all detected levels

In [ ]:
n_stages = len(level_cols) - 1
fig, axes = plt.subplots(1, n_stages, figsize=(6 * n_stages, 7), squeeze=False)
for i, ax in enumerate(axes[0]):
    s, t = level_cols[i], level_cols[i + 1]
    draw_static_sankey(adjacent_flows(df, s, t), title=f'{s} \u2192 {t}', ax=ax)
plt.tight_layout()
combined_png = OUT_DIR / 'sankey_static_combined.png'
fig.savefig(combined_png, dpi=150, bbox_inches='tight')
plt.show()

combined_fig = build_interactive_sankey(df, level_cols, title='Full table-relationship flow')
combined_html = OUT_DIR / 'sankey_interactive_combined.html'
combined_fig.write_html(combined_html)
combined_fig.show()

print('Static  :', combined_png)
print('Interactive:', combined_html)

## Self-check: simulate a deleted level

Drops `level2_table` (if present) from a copy of the data and re-runs the pipeline. Demonstrates the notebook stays functional when the CSV loses a level.

In [ ]:
victim = 'level2_table'
if victim in df.columns:
    df_trimmed = df.drop(columns=[victim])
    trimmed_levels = detect_levels(df_trimmed)
    print('Levels after dropping', victim, '->', trimmed_levels)
    trimmed_fig = build_interactive_sankey(df_trimmed, trimmed_levels, title=f'Flow with {victim} removed')
    trimmed_fig.show()
else:
    print(victim, 'not in CSV; nothing to drop.')